# Playground 04 — Database tables and REAL SQL (no Docker needed)

📖 Primer: [docs/00-concepts.md](../docs/00-concepts.md), section 3

Surprise: Python ships with a tiny real database called **SQLite**. The SQL you type here is the same language you'll use on PostgreSQL in Phase 5.5 — this is the genuine article, just running in memory instead of in a Docker container. We'll build the primer's exact two tables.

> ⚠️ Notebook tip: if you ever see `table bags already exists` or a duplicate-key error after re-running cells, just **Run All** — the first cell creates a fresh empty database each time.

In [ ]:
import sqlite3

db = sqlite3.connect(":memory:")           # a database that lives in RAM
db.execute("PRAGMA foreign_keys = ON")     # make it enforce foreign keys
print("fresh empty database ready")

## 1. CREATE the two tables (the "spreadsheet with rules")

In [ ]:
db.execute("""
    CREATE TABLE bags (
        id        TEXT PRIMARY KEY,   -- unique per row, no exceptions
        brand     TEXT NOT NULL,
        model     TEXT NOT NULL,
        condition TEXT NOT NULL
    )
""")

db.execute("""
    CREATE TABLE price_points (
        id        INTEGER PRIMARY KEY,          -- auto-numbered 1, 2, 3...
        bag_id    TEXT NOT NULL REFERENCES bags(id),   -- the FOREIGN KEY
        price     REAL NOT NULL,
        timestamp TEXT NOT NULL
    )
""")
print("tables 'bags' and 'price_points' created")

## 2. INSERT rows (SQL's word for Create)

In [ ]:
db.execute("INSERT INTO bags VALUES ('chanel-flap-001', 'Chanel', 'Classic Flap', 'like_new')")
db.execute("INSERT INTO bags VALUES ('lv-neverfull-001', 'Louis Vuitton', 'Neverfull MM', 'gently_used')")

# One bag, MANY price points — each row points back via bag_id:
db.execute("INSERT INTO price_points (bag_id, price, timestamp) VALUES ('chanel-flap-001', 9500.0, '2026-01-01')")
db.execute("INSERT INTO price_points (bag_id, price, timestamp) VALUES ('chanel-flap-001', 9800.0, '2026-02-01')")
db.execute("INSERT INTO price_points (bag_id, price, timestamp) VALUES ('lv-neverfull-001', 1800.0, '2026-01-15')")
print("2 bags and 3 price points inserted")

## 3. SELECT — asking the table questions (SQL's Read)

In [ ]:
print("SELECT * FROM bags;")
for row in db.execute("SELECT * FROM bags"):
    print(f"  {row}")

In [ ]:
print("SELECT * FROM bags WHERE brand = 'Chanel';")
for row in db.execute("SELECT * FROM bags WHERE brand = 'Chanel'"):
    print(f"  {row}")

In [ ]:
# All prices ever seen for the Chanel — one-to-many in action:
print("SELECT price, timestamp FROM price_points WHERE bag_id = 'chanel-flap-001';")
for row in db.execute("SELECT price, timestamp FROM price_points WHERE bag_id = 'chanel-flap-001'"):
    print(f"  {row}")

## 4. The rules push back (this is why tables beat spreadsheets)

In [ ]:
# Rule 1: a PRIMARY KEY must be unique. Try inserting a duplicate id:
try:
    db.execute("INSERT INTO bags VALUES ('chanel-flap-001', 'Fake', 'Dupe', 'new')")
except sqlite3.IntegrityError as error:
    print(f"duplicate primary key  -> 💥 refused: {error}")

# Rule 2: a FOREIGN KEY must point at a row that EXISTS:
try:
    db.execute("INSERT INTO price_points (bag_id, price, timestamp) VALUES ('ghost-bag-999', 1.0, '2026-01-01')")
except sqlite3.IntegrityError as error:
    print(f"price for ghost bag    -> 💥 refused: {error}")

print()
print("Bad data stopped AT THE DOOR — no babysitting code needed.")
print("(Phase 5.5's PostgreSQL is even stricter, e.g. about types.)")

## ✏️ Your turn

In [ ]:
# Exercise 1: INSERT a third bag (your dream bag) and give it two
# price points. Then re-run the SELECT cells above and find them.
db.execute("INSERT INTO bags VALUES ('my-dream-bag-001', '...', '...', 'new')")
print("now add two price_points rows for it, then re-run the SELECTs")

In [ ]:
# Exercise 2: PREDICT the output of this query before running it.
for row in db.execute("SELECT model FROM bags WHERE condition = 'gently_used'"):
    print(row)

In [ ]:
# Exercise 3: try inserting a price_point with a missing price —
# replace the price number below with the word NULL (no quotes), run,
# and read which rule refuses.
db.execute("INSERT INTO price_points (bag_id, price, timestamp) VALUES ('chanel-flap-001', 1234.0, '2026-03-01')")
print("inserted fine — now break it as described above")

In [ ]:
# Exercise 4 — the big one: a JOIN stitches the two tables back
# together. Each output row is "a bag and one of its prices, side by
# side." Run it, then try adding bags.model to the SELECT list.
query = """
    SELECT bags.brand, price_points.price
    FROM bags JOIN price_points ON bags.id = price_points.bag_id
"""
for row in db.execute(query):
    print(row)